# 강화학습 2강 — 탐험과 활용 (CartPole · 얼음 호수 · Acrobot)

1강에서는 **무작위로만** 움직였습니다. 오늘은 경험으로 **행동가치(Q)** 를 배워서 더 좋은 행동을 고르게 합니다.

오늘의 순서
1. 경험이 없으면 「가장 좋은 행동(argmax)」을 고를 수가 없다
2. 그래서 **탐험**을 시작한다 — 탐험도 하고, 활용도 하고 (엡실론-그리디)
3. 얼음 호수에서도 — 탐험을 안 하면 영원히 도착 못 한다
4. 그런데 상태가 복잡한 문제(Acrobot)는 이 방법으로 잘 안 풀린다
5. 탐험한 경험을 **기억 상자(리플레이 메모리)** 에 넣고 다시 배운다

> 위에서부터 순서대로 ▶ 실행하세요. (메뉴 「런타임 → 모두 실행」도 됩니다)

## 0. 준비

In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from collections import defaultdict, deque

random.seed(0)   # 매번 같은 결과가 나오게

def 영상으로_보기(화면들, 건너뛰기=1):
    """게임 화면들을 움직이는 영상으로 보여 준다"""
    화면들 = 화면들[::건너뛰기][:300]
    fig = plt.figure(figsize=(4.5, 3))
    그림 = plt.imshow(화면들[0])
    plt.axis("off")
    plt.close(fig)
    영상 = animation.FuncAnimation(fig, lambda i: 그림.set_data(화면들[i]), frames=len(화면들), interval=40)
    display(HTML(영상.to_jshtml()))

# CartPole 의 상태는 숫자 4개: [수레 위치, 수레 속도, 막대 각도, 막대 회전 속도]
# 숫자가 너무 잘게 나뉘어 있어서, 비슷한 상태끼리 「칸」으로 묶는다.
카트폴_경계 = [
    [-0.8, 0.8],                          # 수레 위치  → 3칸
    [-0.5, 0.5],                          # 수레 속도  → 3칸
    np.linspace(-0.21, 0.21, 7)[1:-1],    # 막대 각도  → 6칸
    np.linspace(-2.0, 2.0, 7)[1:-1],      # 막대 회전  → 6칸
]

def 카트폴_칸(state):
    return tuple(int(np.digitize(숫자, 경계)) for 숫자, 경계 in zip(state, 카트폴_경계))

print("준비 완료 · CartPole 칸 개수:", 3 * 3 * 6 * 6)

## 1. 경험이 없으면 argmax 를 할 수 없다

**점수표(Q표)** 는 칸마다 「왼쪽으로 밀면 몇 점, 오른쪽으로 밀면 몇 점」을 적어 둔 표입니다. 이 점수가 **행동가치(Q)** 입니다.
가장 좋은 행동 = 점수가 가장 큰 행동 = `argmax`.

그런데 아직 아무 경험이 없으면? 점수가 전부 0 입니다.

In [ ]:
행동_수 = 2
점수표 = defaultdict(lambda: np.zeros(행동_수))    # 처음 보는 칸은 [0, 0]

env = gym.make("CartPole-v1", render_mode="rgb_array")
state, info = env.reset(seed=1)
칸 = 카트폴_칸(state)
print("지금 칸:", 칸)
print("점수표[칸] =", 점수표[칸], "  ← (왼쪽 점수, 오른쪽 점수)")
print("argmax 가 고른 행동:", int(np.argmax(점수표[칸])), " ← 둘 다 0 이면 언제나 첫 번째(0 = 왼쪽)")

plt.figure(figsize=(4, 2.5))
plt.bar(["left (0)", "right (1)"], 점수표[칸], color=["#888", "#888"])
plt.ylim(-1, 1)
plt.title("Q values with no experience: all 0")
plt.show()

In [ ]:
# 경험 없이 argmax 만으로 한 판
화면들 = [env.render()]
끝 = False
점수 = 0
누른_행동 = []
while not 끝:
    행동 = int(np.argmax(점수표[카트폴_칸(state)]))
    state, reward, terminated, truncated, info = env.step(행동)
    끝 = terminated or truncated
    점수 += reward
    누른_행동.append(행동)
    화면들.append(env.render())
env.close()

print(f"점수: {점수:.0f}점 · 왼쪽 {누른_행동.count(0)}번 / 오른쪽 {누른_행동.count(1)}번")
print("→ 아는 게 없으니 왼쪽만 누르다가 바로 넘어진다")
영상으로_보기(화면들)

## 2. 탐험을 시작한다 — 엡실론-그리디(ε-greedy)

CartPole 같은 문제를 푸는 **고전적인 방법**입니다.

- 확률 **ε(엡실론)** 로 → 아무 행동이나 해 본다 (**탐험**)
- 나머지 확률로 → 점수표에서 가장 좋은 행동(argmax)을 한다 (**활용**)
- 처음엔 ε = 1 (전부 탐험) → 판이 지날수록 조금씩 줄여서 → 마지막엔 거의 활용만

그리고 한 걸음마다 점수표를 고칩니다: 「방금 받은 보상 + 다음 칸에서 기대되는 가장 큰 점수」 쪽으로 조금씩.

In [ ]:
배우는_속도 = 0.1    # 한 번에 얼마나 고칠까
미래_할인 = 0.99     # 미래 점수를 얼마나 믿을까

def 행동_고르기(점수표, 칸, 엡실론, 행동_수):
    if random.random() < 엡실론:
        return random.randrange(행동_수)        # 탐험: 아무거나 해 본다
    return int(np.argmax(점수표[칸]))           # 활용: 지금까지 가장 좋았던 행동

def 점수표_고치기(점수표, 칸, 행동, 보상, 다음_칸, 끝남, 속도=배우는_속도, 할인=미래_할인):
    목표 = 보상 if 끝남 else 보상 + 할인 * np.max(점수표[다음_칸])
    점수표[칸][행동] += 속도 * (목표 - 점수표[칸][행동])

def 학습하기(게임, 칸_찾기, 방식, 판수, 줄이는_비율=0.995, 기억_상자=None, 알림=100,
          최소_엡실론=0.01, 속도=배우는_속도, 할인=미래_할인, 게임_설정={}):
    """방식: '탐험만' / '활용만' / '탐험 → 활용'.  기억_상자를 주면 4번(리플레이 메모리) 방식으로 배운다
    돌려주는 것: 판마다 점수, 가장 잘했을 때(최근 50판 평균이 가장 높을 때)의 점수표"""
    env = gym.make(게임, **게임_설정)
    행동_수 = env.action_space.n
    점수표 = defaultdict(lambda: np.zeros(행동_수))
    엡실론 = {"탐험만": 1.0, "활용만": 0.0}.get(방식, 1.0)
    점수들 = []
    최고_평균, 최고_점수표 = -1e9, {}
    for 판 in range(1, 판수 + 1):
        state, info = env.reset(seed=판)
        칸 = 칸_찾기(state)
        끝 = False
        점수 = 0
        while not 끝:
            행동 = 행동_고르기(점수표, 칸, 엡실론, 행동_수)
            state, reward, terminated, truncated, info = env.step(행동)
            끝 = terminated or truncated
            다음_칸 = 칸_찾기(state)
            if 기억_상자 is None:
                점수표_고치기(점수표, 칸, 행동, reward, 다음_칸, terminated, 속도, 할인)   # 한 번 배우고 버린다
            else:
                기억_상자.append((칸, 행동, reward, 다음_칸, terminated))          # ① 기억 상자에 저장
                if len(기억_상자) >= 64:
                    for 기억 in random.sample(기억_상자, 32):                     # ② 무작위로 꺼내서
                        점수표_고치기(점수표, *기억, 속도, 할인)                    # ③ 다시 배운다
            칸 = 다음_칸
            점수 += reward
        점수들.append(점수)
        if 판 >= 50 and np.mean(점수들[-50:]) > 최고_평균:          # 가장 잘했을 때의 두뇌를 따로 저장
            최고_평균 = np.mean(점수들[-50:])
            최고_점수표 = {칸: 값.copy() for 칸, 값 in 점수표.items()}
        if 방식 == "탐험 → 활용":
            엡실론 = max(최소_엡실론, 엡실론 * 줄이는_비율)   # 탐험을 조금씩 줄인다
        if 판 % 알림 == 0:
            print(f"[{방식}] {판}판 · 탐험 비율 {엡실론:.2f} · 최근 {알림}판 평균 {np.mean(점수들[-알림:]):.0f}점")
    env.close()
    가장_잘한_두뇌 = defaultdict(lambda: np.zeros(행동_수), 최고_점수표)
    return 점수들, 가장_잘한_두뇌

In [ ]:
결과 = {}
for 방식 in ["탐험만", "활용만", "탐험 → 활용"]:
    결과[방식] = 학습하기("CartPole-v1", 카트폴_칸, 방식, 판수=1000, 알림=200)
    print()

print("===== CartPole · 마지막 100판 평균 =====")
for 방식, (점수들, _) in 결과.items():
    print(f"{방식:8s} : {np.mean(점수들[-100:]):6.1f}점")

이름 = {"탐험만": "explore only (e=1)", "활용만": "exploit only (e=0)", "탐험 → 활용": "explore -> exploit"}
plt.figure(figsize=(10, 4))
for 방식, (점수들, _) in 결과.items():
    plt.plot(np.convolve(점수들, np.ones(20) / 20, mode="valid"), label=이름[방식])
plt.xlabel("episode"); plt.ylabel("score (20-episode avg)"); plt.legend(); plt.show()

### Q표 들여다보기 — 탐험을 안 했을 때 vs 했을 때

같은 1000판을 했는데, 두 두뇌의 Q표는 어떻게 다를까요?

In [ ]:
for 방식 in ["활용만", "탐험 → 활용"]:
    점수표 = 결과[방식][1]
    가본_칸 = len(점수표)
    오른쪽_해본_칸 = sum(1 for 값 in 점수표.values() if 값[1] != 0)
    print(f"[{방식}] 가 본 칸 {가본_칸:3d}개 / 324칸 · 그중 오른쪽을 한 번이라도 해 본 칸 {오른쪽_해본_칸}개")

    예시 = list(점수표.items())[:3]
    for 칸, 값 in 예시:
        print(f"     칸 {칸} → 왼쪽 {값[0]:6.1f}점 · 오른쪽 {값[1]:6.1f}점")
    print()

print("→ 탐험을 안 하면 왼쪽만 해 봐서, 오른쪽 점수는 영원히 0 이다. 더 좋은 행동이 있는지 알 수가 없다.")

fig, 축들 = plt.subplots(1, 2, figsize=(10, 4.5))
for 축, 방식 in zip(축들, ["활용만", "탐험 → 활용"]):
    점수표 = 결과[방식][1]
    # 막대 각도(6칸) × 막대 회전(6칸) 지도: 칸마다 더 좋은 행동 (흰색 = 안 가 봄, 파랑 = 왼쪽, 빨강 = 오른쪽)
    지도 = np.full((6, 6), np.nan)
    for (위치, 속도, 각도, 회전), 값 in 점수표.items():
        if 위치 == 1 and 속도 == 1:
            지도[각도, 회전] = np.argmax(값)
    축.imshow(지도, cmap="coolwarm", vmin=0, vmax=1)
    축.set_title({"활용만": "no exploration", "탐험 → 활용": "with exploration"}[방식])
    축.set_xlabel("pole spin  (left <-> right)"); 축.set_ylabel("pole angle")
plt.suptitle("best action per cell   blue = left · red = right · white = never visited")
plt.tight_layout()
plt.show()

In [ ]:
# 「탐험 → 활용」에서 가장 잘했을 때의 두뇌로, 이제는 활용만 해서 5판 → 가장 잘한 판을 영상으로
점수표 = 결과["탐험 → 활용"][1]
env = gym.make("CartPole-v1", render_mode="rgb_array")
최고_점수, 최고_화면 = -1, None
for 판 in range(1, 6):
    state, info = env.reset(seed=100 + 판)
    화면들 = [env.render()]
    끝 = False
    점수 = 0
    while not 끝:
        state, reward, terminated, truncated, info = env.step(int(np.argmax(점수표[카트폴_칸(state)])))
        끝 = terminated or truncated
        점수 += reward
        화면들.append(env.render())
    print(f"{판}판: {점수:.0f}점")
    if 점수 > 최고_점수:
        최고_점수, 최고_화면 = 점수, 화면들
env.close()
print(f"가장 잘한 판({최고_점수:.0f}점)을 영상으로")
영상으로_보기(최고_화면, 건너뛰기=2)

## 3. 얼음 호수(FrozenLake)에서도 똑같다

출발(왼쪽 위)에서 선물(오른쪽 아래)까지 구멍에 빠지지 않고 가면 1점, 그 외에는 0점. 상태는 지금 서 있는 칸 번호(0~15) 하나라 칸을 나눌 필요도 없습니다.
같은 Q표 · 같은 엡실론-그리디로 3000판 — 탐험을 안 했을 때와 했을 때.

In [ ]:
호수_설정 = {"is_slippery": False}          # 미끄럽지 않은 4x4 호수
호수_칸 = lambda state: state              # 상태가 이미 칸 번호

호수 = {}
for 방식 in ["활용만", "탐험 → 활용"]:
    호수[방식] = 학습하기("FrozenLake-v1", 호수_칸, 방식, 판수=3000, 줄이는_비율=0.999, 알림=1000,
                     최소_엡실론=0.05, 속도=0.8, 할인=0.95, 게임_설정=호수_설정)
    print()

def 호수_한판(점수표):
    env = gym.make("FrozenLake-v1", render_mode="rgb_array", **호수_설정)
    state, info = env.reset(seed=0)
    화면들 = [env.render()]
    끝 = False
    보상 = 0
    while not 끝:
        state, 보상, terminated, truncated, info = env.step(int(np.argmax(점수표[state])))
        끝 = terminated or truncated
        화면들.append(env.render())
    env.close()
    return 보상, 화면들

for 방식, (_, 점수표) in 호수.items():
    보상, 화면들 = 호수_한판(점수표)
    print(f"[{방식}] 배운 두뇌로 한 판 → {'도착 ✓' if 보상 > 0 else '실패 ✕'}")
    영상으로_보기(화면들)

## 4. 한계 — 상태가 복잡해지면 (Acrobot)

팔 두 개를 흔들어 끝을 선 위로 올리는 게임입니다. 한 걸음마다 -1점이라 빨리 올릴수록 점수가 높습니다(잘하는 두뇌는 -100점 근처, 끝까지 못 올리면 -500점).
상태가 숫자 **6개** → 같은 방법으로 칸을 나누면 **46,656칸**, CartPole(324칸)의 144배입니다.

In [ ]:
애크로봇_경계 = [np.linspace(-1, 1, 7)[1:-1]] * 4 + [np.linspace(-12.6, 12.6, 7)[1:-1], np.linspace(-28.3, 28.3, 7)[1:-1]]

def 애크로봇_칸(state):
    return tuple(int(np.digitize(숫자, 경계)) for 숫자, 경계 in zip(state, 애크로봇_경계))

print("CartPole 칸 개수 :", 324)
print("Acrobot 칸 개수  :", 6 ** 6)

# CartPole 을 풀었던 그 방법 그대로 300판
애크로봇_점수, 애크로봇_점수표 = 학습하기("Acrobot-v1", 애크로봇_칸, "탐험 → 활용", 판수=300, 줄이는_비율=0.99)
print(f"\n마지막 100판 평균: {np.mean(애크로봇_점수[-100:]):.0f}점  (잘하는 두뇌는 -100점 근처)")
print(f"점수표에 적힌 칸: {len(애크로봇_점수표):,}개 / 전체 {6 ** 6:,}칸")
print("→ 칸이 너무 많아서, 대부분의 칸은 한 번도 못 가 봤다. 처음 보는 칸에서는 아무것도 모른다.")

In [ ]:
# 300판 배운 두뇌로 한 판
env = gym.make("Acrobot-v1", render_mode="rgb_array")
state, info = env.reset(seed=3)
화면들 = [env.render()]
끝 = False
점수 = 0
while not 끝:
    state, reward, terminated, truncated, info = env.step(int(np.argmax(애크로봇_점수표[애크로봇_칸(state)])))
    끝 = terminated or truncated
    점수 += reward
    화면들.append(env.render())
env.close()
print(f"배운 두뇌: {점수:.0f}점 ({len(화면들) - 1}걸음)")
영상으로_보기(화면들, 건너뛰기=3)

## 5. 기억 상자(리플레이 메모리)

지금까지는 한 걸음 걸을 때마다 **한 번 배우고 경험을 버렸습니다.** 탐험으로 어렵게 얻은 경험이 아깝죠.

그래서 경험을 **기억 상자**에 저장해 두고, 무작위로 꺼내서 **여러 번 다시 배웁니다.**

경험 하나 = (지금 칸, 한 행동, 받은 보상, 다음 칸, 끝났는지)

In [ ]:
기억_상자 = deque(maxlen=50000)     # 가득 차면 가장 오래된 기억부터 지워진다

env = gym.make("CartPole-v1")
state, info = env.reset(seed=0)
for _ in range(5):
    행동 = random.randrange(2)
    다음_state, reward, terminated, truncated, info = env.step(행동)
    기억_상자.append((카트폴_칸(state), 행동, reward, 카트폴_칸(다음_state), terminated))   # 저장
    state = 다음_state
env.close()

print("기억 개수:", len(기억_상자))
for 기억 in random.sample(기억_상자, 2):
    print("무작위로 꺼낸 기억:", 기억)

In [ ]:
# CartPole · 같은 300판 동안: 기억 없이 vs 기억 상자
기억없이, _ = 학습하기("CartPole-v1", 카트폴_칸, "탐험 → 활용", 판수=300, 줄이는_비율=0.98, 알림=100)
print()
기억상자, _ = 학습하기("CartPole-v1", 카트폴_칸, "탐험 → 활용", 판수=300, 줄이는_비율=0.98, 알림=100,
                    기억_상자=deque(maxlen=50000))

print("\n===== 100판씩 평균 =====")
print("            1~100판  101~200판  201~300판")
for 이름, 점수들 in [("기억 없이", 기억없이), ("기억 상자", 기억상자)]:
    print(f"{이름} : " + "  ".join(f"{np.mean(점수들[i:i + 100]):8.0f}" for i in (0, 100, 200)))

plt.figure(figsize=(10, 4))
plt.plot(np.convolve(기억없이, np.ones(20) / 20, mode="valid"), label="no memory")
plt.plot(np.convolve(기억상자, np.ones(20) / 20, mode="valid"), label="replay memory")
plt.xlabel("episode"); plt.ylabel("score (20-episode avg)"); plt.legend(); plt.show()
print("→ 같은 판수, 같은 탐험인데 기억 상자 쪽이 더 빨리 잘하게 된다")

## 정리

| 단계 | 결과 |
|---|---|
| ① 경험 없이 argmax | 점수가 전부 0 → 언제나 왼쪽 → 바로 넘어짐 |
| ② 엡실론-그리디 (탐험 → 활용) | CartPole 이 풀린다. 탐험을 안 하면 9점에 갇힌다 |
| ③ 얼음 호수 | 탐험 X → 도착 못 함 · 탐험 O → 도착 |
| ④ 상태가 복잡한 Acrobot | 칸이 4만 6천 개 → 대부분 못 가 봄 → 잘 안 풀린다 |
| ⑤ 기억 상자 (리플레이 메모리) | 같은 탐험으로 더 많이 배운다. 하지만 표로는 한계 |

**다음 3강** — 기억 상자를 더 깊게: 무엇을, 얼마나, 왜 무작위로 꺼내는지.
**4강** — 점수표 대신 **딥러닝**이 처음 보는 칸의 점수도 짐작하게 합니다. 기억 상자 + 딥러닝 = **DQN**.